In [1]:
import torch

print(torch.__version__)

2.7.1+cu118


In [2]:
import os

project_dir = os.path.expanduser("~/work/transformer_chatbot")
data_dir = os.path.join(project_dir, "data")

os.makedirs(data_dir, exist_ok=True)

print("project_dir:", project_dir)
print("data_dir:", data_dir)
print("프로젝트 폴더 존재 여부:", os.path.exists(project_dir))
print("데이터 폴더 존재 여부:", os.path.exists(data_dir))

project_dir: /home/jovyan/work/transformer_chatbot
data_dir: /home/jovyan/work/transformer_chatbot/data
프로젝트 폴더 존재 여부: True
데이터 폴더 존재 여부: True


In [3]:
import urllib.request

url = "https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv"
data_path = os.path.join(data_dir, "ChatbotData.csv")

if not os.path.exists(data_path):
    urllib.request.urlretrieve(url, data_path)
    print("다운로드 완료:", data_path)
else:
    print("이미 파일이 있습니다:", data_path)

print("파일 존재 여부:", os.path.exists(data_path))

다운로드 완료: /home/jovyan/work/transformer_chatbot/data/ChatbotData.csv
파일 존재 여부: True


In [4]:
import pandas as pd

df = pd.read_csv(data_path)

print("데이터 크기:", df.shape)
print("컬럼명:", df.columns.tolist())

df.head()

데이터 크기: (11823, 3)
컬럼명: ['Q', 'A', 'label']


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [5]:
print("결측치 확인")
print(df.isnull().sum())

print("\nlabel 분포")
print(df["label"].value_counts().sort_index())

print("\n질문 길이 통계")
print(df["Q"].astype(str).str.len().describe())

print("\n답변 길이 통계")
print(df["A"].astype(str).str.len().describe())

결측치 확인
Q        0
A        0
label    0
dtype: int64

label 분포
label
0    5290
1    3570
2    2963
Name: count, dtype: int64

질문 길이 통계
count    11823.000000
mean        12.879049
std          6.167467
min          1.000000
25%          9.000000
50%         12.000000
75%         16.000000
max         56.000000
Name: Q, dtype: float64

답변 길이 통계
count    11823.000000
mean        15.015140
std          6.701835
min          1.000000
25%         10.000000
50%         14.000000
75%         18.000000
max         76.000000
Name: A, dtype: float64


In [6]:
import re

def preprocess_sentence(sentence):
    sentence = str(sentence).strip()

    # 구두점 앞뒤에 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)

    # 한글, 자음/모음, 영어, 숫자, 기본 구두점만 남기기
    sentence = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9?.!,]+", " ", sentence)

    # 여러 공백을 하나로 정리
    sentence = re.sub(r"\s+", " ", sentence)

    return sentence.strip()

In [8]:
import re

def preprocess_sentence(sentence):
    sentence = str(sentence).strip()

    # 구두점 앞뒤에 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)

    # 한글, 자음/모음, 영어, 숫자, 기본 구두점만 남기기
    sentence = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9?.!,]+", " ", sentence)

    # 여러 공백을 하나로 정리
    sentence = re.sub(r"\s+", " ", sentence)

    return sentence.strip()

In [9]:
test_sentences = [
    "오늘 너무 힘들어ㅠㅠ",
    "SNS 시간낭비인 거 아는데 매일 하는 중",
    "PPL 심하네!!!",
    "3박4일 놀러가고 싶다",
    "나 잘할 수 있을까?"
]

for s in test_sentences:
    print("원문:", s)
    print("전처리:", preprocess_sentence(s))
    print("-" * 50)

원문: 오늘 너무 힘들어ㅠㅠ
전처리: 오늘 너무 힘들어ㅠㅠ
--------------------------------------------------
원문: SNS 시간낭비인 거 아는데 매일 하는 중
전처리: SNS 시간낭비인 거 아는데 매일 하는 중
--------------------------------------------------
원문: PPL 심하네!!!
전처리: PPL 심하네 ! ! !
--------------------------------------------------
원문: 3박4일 놀러가고 싶다
전처리: 3박4일 놀러가고 싶다
--------------------------------------------------
원문: 나 잘할 수 있을까?
전처리: 나 잘할 수 있을까 ?
--------------------------------------------------


In [10]:
questions = df["Q"].apply(preprocess_sentence).tolist()
answers = df["A"].apply(preprocess_sentence).tolist()

print("전처리된 질문 개수:", len(questions))
print("전처리된 답변 개수:", len(answers))

print("\n예시 5개")
for i in range(5):
    print("Q:", questions[i])
    print("A:", answers[i])
    print("-" * 50)

전처리된 질문 개수: 11823
전처리된 답변 개수: 11823

예시 5개
Q: 12시 땡 !
A: 하루가 또 가네요 .
--------------------------------------------------
Q: 1지망 학교 떨어졌어
A: 위로해 드립니다 .
--------------------------------------------------
Q: 3박4일 놀러가고 싶다
A: 여행은 언제나 좋죠 .
--------------------------------------------------
Q: 3박4일 정도 놀러가고 싶다
A: 여행은 언제나 좋죠 .
--------------------------------------------------
Q: PPL 심하네
A: 눈살이 찌푸려지죠 .
--------------------------------------------------


In [11]:
empty_questions = [q for q in questions if len(q.strip()) == 0]
empty_answers = [a for a in answers if len(a.strip()) == 0]

print("빈 질문 개수:", len(empty_questions))
print("빈 답변 개수:", len(empty_answers))

빈 질문 개수: 0
빈 답변 개수: 0


In [12]:
corpus_path = os.path.join(data_dir, "korean_chatbot_corpus.txt")

with open(corpus_path, "w", encoding="utf-8") as f:
    for q, a in zip(questions, answers):
        f.write(q + "\n")
        f.write(a + "\n")

print("corpus 저장 완료:", corpus_path)
print("파일 존재 여부:", os.path.exists(corpus_path))

corpus 저장 완료: /home/jovyan/work/transformer_chatbot/data/korean_chatbot_corpus.txt
파일 존재 여부: True


In [13]:
with open(corpus_path, "r", encoding="utf-8") as f:
    for i in range(10):
        print(f.readline().strip())

12시 땡 !
하루가 또 가네요 .
1지망 학교 떨어졌어
위로해 드립니다 .
3박4일 놀러가고 싶다
여행은 언제나 좋죠 .
3박4일 정도 놀러가고 싶다
여행은 언제나 좋죠 .
PPL 심하네
눈살이 찌푸려지죠 .


In [15]:
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.2 MB/s eta 0:00:00a 0:00:01


In [16]:
import sentencepiece as spm

print("sentencepiece import 성공")

sentencepiece import 성공


In [17]:
import sentencepiece as spm

vocab_size = 8000
spm_prefix = os.path.join(project_dir, "spm_kor_chatbot")

spm.SentencePieceTrainer.Train(
    input=corpus_path,
    model_prefix=spm_prefix,
    vocab_size=vocab_size,
    model_type="bpe",
    character_coverage=0.9995,
    pad_id=0,
    bos_id=1,
    eos_id=2,
    unk_id=3
)

print("SentencePiece 모델 학습 완료")
print("model 파일:", spm_prefix + ".model")
print("vocab 파일:", spm_prefix + ".vocab")

SentencePiece 모델 학습 완료
model 파일: /home/jovyan/work/transformer_chatbot/spm_kor_chatbot.model
vocab 파일: /home/jovyan/work/transformer_chatbot/spm_kor_chatbot.vocab


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /home/jovyan/work/transformer_chatbot/data/korean_chatbot_corpus.txt
  input_format: 
  model_prefix: /home/jovyan/work/transformer_chatbot/spm_kor_chatbot
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>


In [21]:
sp = spm.SentencePieceProcessor()
sp.Load(spm_prefix + ".model")

print("토크나이저 로드 완료")
print("vocab size:", sp.GetPieceSize())
print("pad id:", sp.pad_id())
print("bos id:", sp.bos_id())
print("eos id:", sp.eos_id())
print("unk id:", sp.unk_id())

토크나이저 로드 완료
vocab size: 8000
pad id: 0
bos id: 1
eos id: 2
unk id: 3


In [22]:
sample = "오늘 너무 힘들어ㅠㅠ"
sample = preprocess_sentence(sample)

pieces = sp.encode(sample, out_type=str)
ids = sp.encode(sample, out_type=int)
decoded = sp.decode(ids)

print("전처리 문장:", sample)
print("pieces:", pieces)
print("ids:", ids)
print("decoded:", decoded)

전처리 문장: 오늘 너무 힘들어ㅠㅠ
pieces: ['▁오늘', '▁너무', '▁힘들어', 'ᅲᅲ']
ids: [129, 56, 418, 2329]
decoded: 오늘 너무 힘들어ᅲᅲ


In [23]:
import torch
from torch.utils.data import Dataset, DataLoader

class ChatbotDataset(Dataset):
    def __init__(self, questions, answers, sp, max_length=40):
        self.questions = questions
        self.answers = answers
        self.sp = sp
        self.max_length = max_length

        self.pad_id = sp.pad_id()
        self.bos_id = sp.bos_id()
        self.eos_id = sp.eos_id()

    def encode_sentence(self, sentence):
        # BOS + 문장 token ids + EOS
        ids = [self.bos_id] + self.sp.encode(sentence, out_type=int) + [self.eos_id]

        # max_length보다 길면 자르기
        ids = ids[:self.max_length]

        # max_length보다 짧으면 PAD로 채우기
        padding = [self.pad_id] * (self.max_length - len(ids))
        ids = ids + padding

        return ids

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        question = self.questions[idx]
        answer = self.answers[idx]

        encoder_input = self.encode_sentence(question)
        answer_ids = self.encode_sentence(answer)

        # 디코더 입력과 정답을 한 칸 shift
        decoder_input = answer_ids[:-1]
        decoder_label = answer_ids[1:]

        return (
            torch.tensor(encoder_input, dtype=torch.long),
            torch.tensor(decoder_input, dtype=torch.long),
            torch.tensor(decoder_label, dtype=torch.long)
        )

In [24]:
MAX_LENGTH = 40
BATCH_SIZE = 32

dataset = ChatbotDataset(
    questions=questions,
    answers=answers,
    sp=sp,
    max_length=MAX_LENGTH
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Dataset 크기:", len(dataset))
print("DataLoader batch 개수:", len(dataloader))

Dataset 크기: 11823
DataLoader batch 개수: 370


In [25]:
encoder_input, decoder_input, decoder_label = dataset[0]

print("encoder_input 크기:", encoder_input.size())
print("decoder_input 크기:", decoder_input.size())
print("decoder_label 크기:", decoder_label.size())

print("\nencoder_input:", encoder_input)
print("\ndecoder_input:", decoder_input)
print("\ndecoder_label:", decoder_label)

print("\n질문 복원:")
print(sp.decode([int(x) for x in encoder_input if int(x) not in [sp.pad_id(), sp.bos_id(), sp.eos_id()]]))

print("\n디코더 입력 복원:")
print(sp.decode([int(x) for x in decoder_input if int(x) not in [sp.pad_id(), sp.bos_id(), sp.eos_id()]]))

print("\n디코더 정답 복원:")
print(sp.decode([int(x) for x in decoder_label if int(x) not in [sp.pad_id(), sp.bos_id(), sp.eos_id()]]))

encoder_input 크기: torch.Size([40])
decoder_input 크기: torch.Size([39])
decoder_label 크기: torch.Size([39])

encoder_input: tensor([   1, 5560, 6966, 3207,  108,    2,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])

decoder_input: tensor([   1, 4492,  214, 5930,    4,    2,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0])

decoder_label: tensor([4492,  214, 5930,    4,    2,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0])

질문 복원:
12시 땡 !

디코더 입력 복원:
하루가 또 가네요 .


In [26]:
for encoder_input, decoder_input, decoder_label in dataloader:
    print("encoder_input batch 크기:", encoder_input.size())
    print("decoder_input batch 크기:", decoder_input.size())
    print("decoder_label batch 크기:", decoder_label.size())
    break

encoder_input batch 크기: torch.Size([32, 40])
decoder_input batch 크기: torch.Size([32, 39])
decoder_label batch 크기: torch.Size([32, 39])


In [27]:
needed_names = [
    "PositionalEncoding",
    "scaled_dot_product_attention",
    "MultiHeadAttention",
    "EncoderLayer",
    "Encoder",
    "DecoderLayer",
    "Decoder",
    "Transformer",
    "create_padding_mask",
    "create_look_ahead_mask"
]

for name in needed_names:
    print(name, "정의 여부:", name in globals())

PositionalEncoding 정의 여부: False
scaled_dot_product_attention 정의 여부: False
MultiHeadAttention 정의 여부: False
EncoderLayer 정의 여부: False
Encoder 정의 여부: False
DecoderLayer 정의 여부: False
Decoder 정의 여부: False
Transformer 정의 여부: False
create_padding_mask 정의 여부: False
create_look_ahead_mask 정의 여부: False


In [28]:
NUM_LAYERS = 2
D_MODEL = 256
NUM_HEADS = 8
UNITS = 512
DROPOUT = 0.1
VOCAB_SIZE = sp.GetPieceSize()

print("VOCAB_SIZE:", VOCAB_SIZE)

VOCAB_SIZE: 8000


In [30]:
needed_names = [
    "PositionalEncoding",
    "scaled_dot_product_attention",
    "MultiHeadAttention",
    "EncoderLayer",
    "Encoder",
    "DecoderLayer",
    "Decoder",
    "Transformer",
    "create_padding_mask",
    "create_look_ahead_mask"
]

for name in needed_names:
    print(name, "정의 여부:", name in globals())

PositionalEncoding 정의 여부: False
scaled_dot_product_attention 정의 여부: False
MultiHeadAttention 정의 여부: False
EncoderLayer 정의 여부: False
Encoder 정의 여부: False
DecoderLayer 정의 여부: False
Decoder 정의 여부: False
Transformer 정의 여부: False
create_padding_mask 정의 여부: False
create_look_ahead_mask 정의 여부: False


In [31]:
import torch
import torch.nn as nn

class Transformer(nn.Module):
    def __init__(self,
                 vocab_size,
                 num_layers,
                 units,
                 d_model,
                 num_heads,
                 dropout=0.1):
        super(Transformer, self).__init__()

        self.encoder = Encoder(
            vocab_size=vocab_size,
            num_layers=num_layers,
            ff_dim=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        self.decoder = Decoder(
            vocab_size=vocab_size,
            num_layers=num_layers,
            ff_dim=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        self.final_linear = nn.Linear(d_model, vocab_size)

    def forward(self, inputs, dec_inputs):
        enc_padding_mask = create_padding_mask(inputs)
        look_ahead_mask = create_look_ahead_mask(dec_inputs)
        dec_padding_mask = create_padding_mask(inputs)

        enc_outputs = self.encoder(
            x=inputs,
            mask=enc_padding_mask
        )

        dec_outputs = self.decoder(
            x=dec_inputs,
            enc_outputs=enc_outputs,
            look_ahead_mask=look_ahead_mask,
            padding_mask=dec_padding_mask
        )

        logits = self.final_linear(dec_outputs)

        return logits

In [32]:
print("Transformer 정의 여부:", "Transformer" in globals())

Transformer 정의 여부: True


In [153]:
def decoder_inference(model, sentence, tokenizer, device='cpu', input_max_length=100, output_max_length=110):
    START_TOKEN = tokenizer.bos_id()
    END_TOKEN = tokenizer.eos_id()
    PAD_TOKEN = tokenizer.pad_id()

    sentence = preprocess_sentence(sentence)

    enc_input_ids = [START_TOKEN] + tokenizer.encode(sentence, out_type=int) + [END_TOKEN]
    enc_input_ids = enc_input_ids[:input_max_length]

    enc_input = torch.tensor([enc_input_ids], dtype=torch.long, device=device)
    dec_input = torch.tensor([[START_TOKEN]], dtype=torch.long, device=device)

    model.eval()

    with torch.no_grad():
        for _ in range(output_max_length):
            logits = model(enc_input, dec_input)
            last_step_logits = logits[:, -1, :]
            predicted_id = torch.argmax(last_step_logits, dim=-1)

            if predicted_id.item() == END_TOKEN:
                break

            predicted_id = predicted_id.unsqueeze(0)
            dec_input = torch.cat([dec_input, predicted_id], dim=1)

            if dec_input.size(1) >= output_max_length:
                break

    output_sequence = dec_input.squeeze(0).tolist()

    output_sequence = [
        token for token in output_sequence
        if token not in [START_TOKEN, END_TOKEN, PAD_TOKEN]
    ]

    return output_sequence


def sentence_generation(model, sentence, tokenizer, device='cpu'):
    output_seq = decoder_inference(
        model=model,
        sentence=sentence,
        tokenizer=tokenizer,
        device=device,
        input_max_length=100,
        output_max_length=110
    )

    predicted_sentence = tokenizer.decode(output_seq)

    print("입력:", sentence)
    print("출력:", predicted_sentence)
    print("-" * 80)

    return predicted_sentence

In [34]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [35]:
class PositionalEncoding(nn.Module):
    def __init__(self, position, d_model):
        super(PositionalEncoding, self).__init__()

        self.pos_encoding = self.positional_encoding(position, d_model)

    def get_angles(self, position, i, d_model):
        angles = 1 / torch.pow(
            torch.tensor(10000.0),
            (2 * (i // 2)) / torch.tensor(float(d_model))
        )
        return position * angles

    def positional_encoding(self, position, d_model):
        angle_rads = self.get_angles(
            torch.arange(position).unsqueeze(1),
            torch.arange(d_model).unsqueeze(0),
            d_model
        )

        angle_rads[:, 0::2] = torch.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = torch.cos(angle_rads[:, 1::2])

        pos_encoding = angle_rads.unsqueeze(0)
        return pos_encoding.float()

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pos_encoding[:, :seq_len, :].to(x.device)

In [154]:
def scaled_dot_product_attention(query, key, value, mask=None):
    matmul_qk = torch.matmul(query, key.transpose(-1, -2))

    depth = key.size(-1)
    logits = matmul_qk / math.sqrt(depth)

    if mask is not None:
        logits += (mask * -1e9)

    attention_weights = F.softmax(logits, dim=-1)
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

In [37]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.query_dense = nn.Linear(d_model, d_model)
        self.key_dense = nn.Linear(d_model, d_model)
        self.value_dense = nn.Linear(d_model, d_model)

        self.out_dense = nn.Linear(d_model, d_model)

    def split_heads(self, inputs):
        batch_size = inputs.size(0)

        inputs = inputs.view(batch_size, -1, self.num_heads, self.depth)
        return inputs.transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)

        query = self.split_heads(query)
        key = self.split_heads(key)
        value = self.split_heads(value)

        scaled_attention, attention_weights = scaled_dot_product_attention(
            query, key, value, mask
        )

        scaled_attention = scaled_attention.transpose(1, 2)

        batch_size = scaled_attention.size(0)
        concat_attention = scaled_attention.contiguous().view(
            batch_size, -1, self.d_model
        )

        outputs = self.out_dense(concat_attention)

        return outputs

In [38]:
def create_padding_mask(x):
    mask = (x == 0).float()
    mask = mask.unsqueeze(1).unsqueeze(2)
    return mask


def create_look_ahead_mask(x):
    seq_len = x.size(1)

    look_ahead_mask = 1 - torch.tril(torch.ones((seq_len, seq_len)))
    look_ahead_mask = look_ahead_mask.to(x.device)

    padding_mask = create_padding_mask(x)

    look_ahead_mask = look_ahead_mask.unsqueeze(0).unsqueeze(1)

    combined_mask = torch.max(look_ahead_mask, padding_mask)

    return combined_mask

In [39]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(EncoderLayer, self).__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )

        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output)

        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)

        out2 = self.layernorm2(out1 + ffn_output)

        return out2


class Encoder(nn.Module):
    def __init__(self, vocab_size, num_layers, ff_dim, d_model, num_heads, dropout=0.1):
        super(Encoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(1000, d_model)

        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.embedding(x)
        x *= math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        for i in range(self.num_layers):
            x = self.enc_layers[i](x, mask)

        return x

In [40]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(DecoderLayer, self).__init__()

        self.self_mha = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_mha = MultiHeadAttention(d_model, num_heads)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )

        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_outputs, look_ahead_mask, padding_mask):
        attn1 = self.self_mha(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1)
        out1 = self.layernorm1(x + attn1)

        attn2 = self.enc_dec_mha(out1, enc_outputs, enc_outputs, padding_mask)
        attn2 = self.dropout2(attn2)
        out2 = self.layernorm2(out1 + attn2)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output)
        out3 = self.layernorm3(out2 + ffn_output)

        return out3


class Decoder(nn.Module):
    def __init__(self, vocab_size, num_layers, ff_dim, d_model, num_heads, dropout=0.1):
        super(Decoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(1000, d_model)

        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_outputs, look_ahead_mask, padding_mask):
        x = self.embedding(x)
        x *= math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        for i in range(self.num_layers):
            x = self.dec_layers[i](
                x,
                enc_outputs,
                look_ahead_mask,
                padding_mask
            )

        return x

In [41]:
class Transformer(nn.Module):
    def __init__(self,
                 vocab_size,
                 num_layers,
                 units,
                 d_model,
                 num_heads,
                 dropout=0.1):
        super(Transformer, self).__init__()

        self.encoder = Encoder(
            vocab_size=vocab_size,
            num_layers=num_layers,
            ff_dim=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        self.decoder = Decoder(
            vocab_size=vocab_size,
            num_layers=num_layers,
            ff_dim=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        self.final_linear = nn.Linear(d_model, vocab_size)

    def forward(self, inputs, dec_inputs):
        enc_padding_mask = create_padding_mask(inputs)
        look_ahead_mask = create_look_ahead_mask(dec_inputs)
        dec_padding_mask = create_padding_mask(inputs)

        enc_outputs = self.encoder(inputs, enc_padding_mask)

        dec_outputs = self.decoder(
            dec_inputs,
            enc_outputs,
            look_ahead_mask,
            dec_padding_mask
        )

        logits = self.final_linear(dec_outputs)

        return logits

In [42]:
needed_names = [
    "PositionalEncoding",
    "scaled_dot_product_attention",
    "MultiHeadAttention",
    "EncoderLayer",
    "Encoder",
    "DecoderLayer",
    "Decoder",
    "Transformer",
    "create_padding_mask",
    "create_look_ahead_mask"
]

for name in needed_names:
    print(name, "정의 여부:", name in globals())

PositionalEncoding 정의 여부: True
scaled_dot_product_attention 정의 여부: True
MultiHeadAttention 정의 여부: True
EncoderLayer 정의 여부: True
Encoder 정의 여부: True
DecoderLayer 정의 여부: True
Decoder 정의 여부: True
Transformer 정의 여부: True
create_padding_mask 정의 여부: True
create_look_ahead_mask 정의 여부: True


In [43]:
model = Transformer(
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS,
    units=UNITS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dropout=DROPOUT
)

print("모델 생성 완료")
print(model)

모델 생성 완료
Transformer(
  (encoder): Encoder(
    (embedding): Embedding(8000, 256)
    (pos_encoding): PositionalEncoding()
    (enc_layers): ModuleList(
      (0-1): 2 x EncoderLayer(
        (mha): MultiHeadAttention(
          (query_dense): Linear(in_features=256, out_features=256, bias=True)
          (key_dense): Linear(in_features=256, out_features=256, bias=True)
          (value_dense): Linear(in_features=256, out_features=256, bias=True)
          (out_dense): Linear(in_features=256, out_features=256, bias=True)
        )
        (ffn): Sequential(
          (0): Linear(in_features=256, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=256, bias=True)
        )
        (layernorm1): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
        (layernorm2): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  

In [44]:
for encoder_input, decoder_input, decoder_label in dataloader:
    sample_logits = model(encoder_input, decoder_input)
    print("sample_logits 크기:", sample_logits.size())
    break

sample_logits 크기: torch.Size([32, 39, 8000])


In [45]:
loss_function = torch.nn.CrossEntropyLoss(ignore_index=sp.pad_id())

print("Loss 함수 생성 완료")
print("pad_id:", sp.pad_id())

Loss 함수 생성 완료
pad_id: 0


In [46]:
def get_lr_lambda(d_model, warmup_steps=4000):
    d_model = float(d_model)

    def lr_lambda(step):
        step = step + 1
        return (d_model ** -0.5) * min(
            step ** -0.5,
            step * (warmup_steps ** -1.5)
        )

    return lr_lambda

print("학습률 스케줄러 함수 정의 완료")

학습률 스케줄러 함수 정의 완료


In [47]:
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler

optimizer = optim.Adam(
    model.parameters(),
    lr=1.0,
    betas=(0.9, 0.98),
    eps=1e-9
)

scheduler = lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=get_lr_lambda(D_MODEL, warmup_steps=4000)
)

print("Optimizer / Scheduler 생성 완료")

Optimizer / Scheduler 생성 완료


In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("device:", device)

device: cuda


In [49]:
def accuracy_function(y_pred, y_true, pad_id=0):
    """
    y_pred: (batch_size, seq_len, vocab_size)
    y_true: (batch_size, seq_len)
    """
    preds = y_pred.argmax(dim=-1)
    mask = (y_true != pad_id)

    correct = (preds == y_true) & mask
    acc = correct.float().sum() / mask.float().sum()

    return acc

In [50]:
def train_step(model, batch, optimizer, loss_function, device):
    model.train()

    enc_input, dec_input, target = [x.to(device) for x in batch]

    optimizer.zero_grad()

    logits = model(enc_input, dec_input)

    # CrossEntropyLoss는 class 차원(vocab_size)이 가운데 있어야 함
    loss = loss_function(logits.permute(0, 2, 1), target)

    loss.backward()

    # gradient 폭주 방지
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    acc = accuracy_function(logits, target, pad_id=sp.pad_id())

    return loss.item(), acc.item()

In [51]:
def train(model, dataloader, optimizer, loss_function, scheduler, num_epochs, device):
    model.to(device)

    for epoch in range(num_epochs):
        total_loss = 0
        total_acc = 0

        for step, batch in enumerate(dataloader):
            loss, acc = train_step(
                model,
                batch,
                optimizer,
                loss_function,
                device
            )

            total_loss += loss
            total_acc += acc

            scheduler.step()

            if step % 50 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(
                    f"[Epoch {epoch+1}, Step {step}] "
                    f"Loss: {loss:.4f}, Acc: {acc:.4f}, LR: {current_lr:.8f}"
                )

        avg_loss = total_loss / len(dataloader)
        avg_acc = total_acc / len(dataloader)

        print(
            f"Epoch {epoch+1} Completed - "
            f"Avg Loss: {avg_loss:.4f}, Avg Acc: {avg_acc:.4f}"
        )

In [52]:
%%time

train(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    loss_function=loss_function,
    scheduler=scheduler,
    num_epochs=1,
    device=device
)

[Epoch 1, Step 0] Loss: 9.0235, Acc: 0.0000, LR: 0.00000049
[Epoch 1, Step 50] Loss: 8.5414, Acc: 0.0474, LR: 0.00001285
[Epoch 1, Step 100] Loss: 7.5087, Acc: 0.2871, LR: 0.00002520
[Epoch 1, Step 150] Loss: 6.7729, Acc: 0.2968, LR: 0.00003755
[Epoch 1, Step 200] Loss: 6.6156, Acc: 0.2739, LR: 0.00004990
[Epoch 1, Step 250] Loss: 6.3215, Acc: 0.2616, LR: 0.00006226
[Epoch 1, Step 300] Loss: 6.0978, Acc: 0.3029, LR: 0.00007461
[Epoch 1, Step 350] Loss: 6.0690, Acc: 0.2851, LR: 0.00008696
Epoch 1 Completed - Avg Loss: 6.9881, Avg Acc: 0.2417
CPU times: user 16.5 s, sys: 307 ms, total: 16.8 s
Wall time: 11.4 s


In [53]:
%%time

train(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    loss_function=loss_function,
    scheduler=scheduler,
    num_epochs=4,
    device=device
)

[Epoch 1, Step 0] Loss: 5.9080, Acc: 0.2723, LR: 0.00009190
[Epoch 1, Step 50] Loss: 5.5780, Acc: 0.2977, LR: 0.00010426
[Epoch 1, Step 100] Loss: 5.8797, Acc: 0.2731, LR: 0.00011661
[Epoch 1, Step 150] Loss: 5.7185, Acc: 0.3053, LR: 0.00012896
[Epoch 1, Step 200] Loss: 5.8632, Acc: 0.2791, LR: 0.00014131
[Epoch 1, Step 250] Loss: 5.2896, Acc: 0.3259, LR: 0.00015367
[Epoch 1, Step 300] Loss: 5.5333, Acc: 0.3077, LR: 0.00016602
[Epoch 1, Step 350] Loss: 5.5963, Acc: 0.2960, LR: 0.00017837
Epoch 1 Completed - Avg Loss: 5.6563, Avg Acc: 0.3014
[Epoch 2, Step 0] Loss: 5.6190, Acc: 0.2708, LR: 0.00018331
[Epoch 2, Step 50] Loss: 5.5730, Acc: 0.2915, LR: 0.00019567
[Epoch 2, Step 100] Loss: 5.0734, Acc: 0.3364, LR: 0.00020802
[Epoch 2, Step 150] Loss: 5.5000, Acc: 0.3125, LR: 0.00022037
[Epoch 2, Step 200] Loss: 5.0302, Acc: 0.3226, LR: 0.00023272
[Epoch 2, Step 250] Loss: 4.8326, Acc: 0.3541, LR: 0.00024508
[Epoch 2, Step 300] Loss: 5.1311, Acc: 0.3300, LR: 0.00025743
[Epoch 2, Step 350] Lo

In [54]:
model_save_path = os.path.join(project_dir, "korean_transformer_chatbot.pt")

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "vocab_size": VOCAB_SIZE,
    "num_layers": NUM_LAYERS,
    "d_model": D_MODEL,
    "num_heads": NUM_HEADS,
    "units": UNITS,
    "dropout": DROPOUT,
}, model_save_path)

print("모델 저장 완료:", model_save_path)
print("파일 존재 여부:", os.path.exists(model_save_path))

모델 저장 완료: /home/jovyan/work/transformer_chatbot/korean_transformer_chatbot.pt
파일 존재 여부: True


In [145]:
def decoder_inference(model, sentence, tokenizer, device='cpu', max_length=40):
    START_TOKEN = tokenizer.bos_id()
    END_TOKEN = tokenizer.eos_id()
    PAD_TOKEN = tokenizer.pad_id()

    # 1. 입력 문장 전처리
    sentence = preprocess_sentence(sentence)

    # 2. 입력 문장 인코딩
    enc_input_ids = [START_TOKEN] + tokenizer.encode(sentence, out_type=int) + [END_TOKEN]
    enc_input_ids = enc_input_ids[:max_length]

    # 3. 인코더 입력 텐서 생성
    enc_input = torch.tensor([enc_input_ids], dtype=torch.long, device=device)

    # 4. 디코더 입력은 START_TOKEN에서 시작
    dec_input = torch.tensor([[START_TOKEN]], dtype=torch.long, device=device)

    model.eval()

    with torch.no_grad():
        for _ in range(max_length):
            logits = model(enc_input, dec_input)

            # 마지막 위치의 vocab 점수만 사용
            last_step_logits = logits[:, -1, :]

            # 가장 확률 높은 토큰 선택
            predicted_id = torch.argmax(last_step_logits, dim=-1)

            # END_TOKEN 나오면 종료
            if predicted_id.item() == END_TOKEN:
                break

            # 예측 토큰을 디코더 입력 뒤에 붙이기
            predicted_id = predicted_id.unsqueeze(0)
            dec_input = torch.cat([dec_input, predicted_id], dim=1)

            if dec_input.size(1) >= max_length:
                break

    output_sequence = dec_input.squeeze(0).tolist()

    # START, END, PAD 제거
    output_sequence = [
        token for token in output_sequence
        if token not in [START_TOKEN, END_TOKEN, PAD_TOKEN]
    ]

    return output_sequence

In [156]:
%%time

train(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    loss_function=loss_function,
    scheduler=scheduler,
    num_epochs=20,
    device=device
)

[Epoch 1, Step 0] Loss: 0.0095, Acc: 1.0000, LR: 0.00024920
[Epoch 1, Step 50] Loss: 0.0244, Acc: 0.9955, LR: 0.00024910
[Epoch 1, Step 100] Loss: 0.0220, Acc: 0.9956, LR: 0.00024900
[Epoch 1, Step 150] Loss: 0.0351, Acc: 0.9960, LR: 0.00024890
[Epoch 1, Step 200] Loss: 0.0637, Acc: 0.9871, LR: 0.00024880
[Epoch 1, Step 250] Loss: 0.0412, Acc: 0.9952, LR: 0.00024871
[Epoch 1, Step 300] Loss: 0.0442, Acc: 0.9951, LR: 0.00024861
[Epoch 1, Step 350] Loss: 0.0624, Acc: 0.9902, LR: 0.00024851
Epoch 1 Completed - Avg Loss: 0.0374, Avg Acc: 0.9954
[Epoch 2, Step 0] Loss: 0.0086, Acc: 1.0000, LR: 0.00024847
[Epoch 2, Step 50] Loss: 0.0155, Acc: 1.0000, LR: 0.00024837
[Epoch 2, Step 100] Loss: 0.0174, Acc: 1.0000, LR: 0.00024827
[Epoch 2, Step 150] Loss: 0.0594, Acc: 0.9906, LR: 0.00024818
[Epoch 2, Step 200] Loss: 0.0362, Acc: 0.9954, LR: 0.00024808
[Epoch 2, Step 250] Loss: 0.0451, Acc: 0.9952, LR: 0.00024798
[Epoch 2, Step 300] Loss: 0.0180, Acc: 1.0000, LR: 0.00024788
[Epoch 2, Step 350] Lo

In [144]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "vocab_size": VOCAB_SIZE,
    "num_layers": NUM_LAYERS,
    "d_model": D_MODEL,
    "num_heads": NUM_HEADS,
    "units": UNITS,
    "dropout": DROPOUT,
}, model_save_path)

print("20 epoch 추가 학습 후 모델 저장 완료:", model_save_path)
print("파일 존재 여부:", os.path.exists(model_save_path))

20 epoch 추가 학습 후 모델 저장 완료: /home/jovyan/work/transformer_chatbot/korean_transformer_chatbot.pt
파일 존재 여부: True


In [146]:
%%time

train(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    loss_function=loss_function,
    scheduler=scheduler,
    num_epochs=10,
    device=device
)

[Epoch 1, Step 0] Loss: 0.0101, Acc: 1.0000, LR: 0.00025687
[Epoch 1, Step 50] Loss: 0.0103, Acc: 1.0000, LR: 0.00025676
[Epoch 1, Step 100] Loss: 0.0225, Acc: 1.0000, LR: 0.00025665
[Epoch 1, Step 150] Loss: 0.0198, Acc: 1.0000, LR: 0.00025654
[Epoch 1, Step 200] Loss: 0.0457, Acc: 0.9904, LR: 0.00025644
[Epoch 1, Step 250] Loss: 0.0308, Acc: 0.9959, LR: 0.00025633
[Epoch 1, Step 300] Loss: 0.0723, Acc: 0.9862, LR: 0.00025622
[Epoch 1, Step 350] Loss: 0.0261, Acc: 1.0000, LR: 0.00025611
Epoch 1 Completed - Avg Loss: 0.0381, Avg Acc: 0.9953
[Epoch 2, Step 0] Loss: 0.0119, Acc: 1.0000, LR: 0.00025607
[Epoch 2, Step 50] Loss: 0.0115, Acc: 1.0000, LR: 0.00025596
[Epoch 2, Step 100] Loss: 0.0143, Acc: 1.0000, LR: 0.00025586
[Epoch 2, Step 150] Loss: 0.0834, Acc: 0.9865, LR: 0.00025575
[Epoch 2, Step 200] Loss: 0.0068, Acc: 1.0000, LR: 0.00025564
[Epoch 2, Step 250] Loss: 0.0629, Acc: 0.9903, LR: 0.00025553
[Epoch 2, Step 300] Loss: 0.0451, Acc: 0.9947, LR: 0.00025543
[Epoch 2, Step 350] Lo

In [143]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "vocab_size": VOCAB_SIZE,
    "num_layers": NUM_LAYERS,
    "d_model": D_MODEL,
    "num_heads": NUM_HEADS,
    "units": UNITS,
    "dropout": DROPOUT,
}, model_save_path)

print("10 epoch 추가 학습 후 모델 저장 완료:", model_save_path)
print("파일 존재 여부:", os.path.exists(model_save_path))

10 epoch 추가 학습 후 모델 저장 완료: /home/jovyan/work/transformer_chatbot/korean_transformer_chatbot.pt
파일 존재 여부: True


In [142]:
test_sentences = [
    "오늘 너무 보고싶은 사람이 있는데 어쩌면 좋을까?",
    "나 정말 깊은 사랑을 혼자했던 것 같아",
    "그 사람은 말수는 적지만 내게 집중하려고 노력하는 사람이었어.",
    "혹시 그 사람은 지금 내 생각하고 있을까?",
    "그 사람이 삶을 대하는 태도를 나도 닮고 싶었는데 내가 놓친 걸까?",
    "그 사람과의 관계를 다시 회복할 수 있을까?",
    "하루하루 스스로 신뢰를 적립하는 마음으로 최선을 다하고 있어."
]

test_results = []

for sentence in test_sentences:
    output = sentence_generation(model, sentence, sp, device)
    test_results.append((sentence, output))

result_df = pd.DataFrame(test_results, columns=["Input", "Generated Response"])
result_df

입력: 오늘 너무 보고싶은 사람이 있는데 어쩌면 좋을까?
출력: 천천히 다가가보세요 .
--------------------------------------------------------------------------------
입력: 나 정말 깊은 사랑을 혼자했던 것 같아
출력: 제가 당신을 좋아하고 있어요 .
--------------------------------------------------------------------------------
입력: 그 사람은 말수는 적지만 내게 집중하려고 노력하는 사람이었어.
출력: 맘 고생 많았어요 .
--------------------------------------------------------------------------------
입력: 혹시 그 사람은 지금 내 생각하고 있을까?
출력: 조금 후련해질 거예요 .
--------------------------------------------------------------------------------
입력: 그 사람이 삶을 대하는 태도를 나도 닮고 싶었는데 내가 놓친 걸까?
출력: 생각을 정리해 보는 건 어떨까요 .
--------------------------------------------------------------------------------
입력: 그 사람과의 관계를 다시 회복할 수 있을까?
출력: 새로운 데이트 코스를 찾아보세요 .
--------------------------------------------------------------------------------
입력: 하루하루 스스로 신뢰를 적립하는 마음으로 최선을 다하고 있어.
출력: 그런 생각을 들게 하는 사람 상종하지 마세요 .
--------------------------------------------------------------------------------


,Input,Generated Response
0,오늘 너무 보고싶은 사람이 있는데 어쩌면 좋을까?,천천히 다가가보세요 .
1,나 정말 깊은 사랑을 혼자했던 것 같아,제가 당신을 좋아하고 있어요 .
2,그 사람은 말수는 적지만 내게 집중하려고 노력하는 사람이었어.,맘 고생 많았어요 .
3,혹시 그 사람은 지금 내 생각하고 있을까?,조금 후련해질 거예요 .
4,그 사람이 삶을 대하는 태도를 나도 닮고 싶었는데 내가 놓친 걸까?,생각을 정리해 보는 건 어떨까요 .
5,그 사람과의 관계를 다시 회복할 수 있을까?,새로운 데이트 코스를 찾아보세요 .
6,하루하루 스스로 신뢰를 적립하는 마음으로 최선을 다하고 있어.,그런 생각을 들게 하는 사람 상종하지 마세요 .


In [155]:
test_sentences = [
    "오늘 날씨가 너무 흐리고 안좋네. 넌 어때?",
    "FOMO로 인해서 코딩 공부를 하는데 잘 안되네.",
    "너는 인문학 전공자이자 코딩 입문자가 이런 실습 수업 수강하는 거 어떻게 생각해?",
    "공부하는데 도움되는 행동이 좋을까? 불필요한 행동을 제거하는 게 좋을까?",
    "너는 나의 학습법에 대해서 어떻게 생각해?",
    "매일 적립하는 마음으로 학습하지만 제한된 시간 내에 집중하려니 어렵네."
]

test_results = []

for sentence in test_sentences:
    output = sentence_generation(model, sentence, sp, device)
    test_results.append((sentence, output))

result_df = pd.DataFrame(test_results, columns=["Input", "Generated Response"])
result_df

입력: 오늘 날씨가 너무 흐리고 안좋네. 넌 어때?
출력: 이제 일어날 시간이에요 .
--------------------------------------------------------------------------------
입력: FOMO로 인해서 코딩 공부를 하는데 잘 안되네.
출력: 잘 생각하셨어요 .
--------------------------------------------------------------------------------
입력: 너는 인문학 전공자이자 코딩 입문자가 이런 실습 수업 수강하는 거 어떻게 생각해?
출력: 맛있게 드세요 .
--------------------------------------------------------------------------------
입력: 공부하는데 도움되는 행동이 좋을까? 불필요한 행동을 제거하는 게 좋을까?
출력: 충분히 그럴 수 있어요 .
--------------------------------------------------------------------------------
입력: 너는 나의 학습법에 대해서 어떻게 생각해?
출력: 사탕 만들어요 .
--------------------------------------------------------------------------------
입력: 매일 적립하는 마음으로 학습하지만 제한된 시간 내에 집중하려니 어렵네.
출력: 할 순 있어요 .
--------------------------------------------------------------------------------


,Input,Generated Response
0,오늘 날씨가 너무 흐리고 안좋네. 넌 어때?,이제 일어날 시간이에요 .
1,FOMO로 인해서 코딩 공부를 하는데 잘 안되네.,잘 생각하셨어요 .
2,너는 인문학 전공자이자 코딩 입문자가 이런 실습 수업 수강하는 거 어떻게 생각해?,맛있게 드세요 .
3,공부하는데 도움되는 행동이 좋을까? 불필요한 행동을 제거하는 게 좋을까?,충분히 그럴 수 있어요 .
4,너는 나의 학습법에 대해서 어떻게 생각해?,사탕 만들어요 .
5,매일 적립하는 마음으로 학습하지만 제한된 시간 내에 집중하려니 어렵네.,할 순 있어요 .
